# E2.3 · Voluntary frameworks as your spine

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.2 · Horizontal AI regulation](https://spbreed.github.io/cyber-commons/lessons/E2.2.html)**.

| | |
|---|---|
| Tools used | NIST AI RMF, OSCAL |

## What this lesson is

**What it covers.** Hang two regulator mappings off one framework spine.

**Why a security engineer needs it.** Regime-specific mappings with nothing to hang off. The control it builds is: aI RMF / management-system standards as the structure; regulator mappings as overlays.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Voluntary frameworks are the cheapest structural decision available: build one control set against a recognised spine, then map it outward to every regime that asks. The alternative is a control set per regulator.

> **At CyberTravels.** One control set for CyberTravels, mapped outward to every regime that asks — rather than a control set per regulator, which is where CyberTravels would otherwise end up.

## 2 · The framework

```
   one control set, mapped outward

                 +--------------------+
                 |  your control set  |
                 +---------+----------+
                           |
        +------------+-----+------+------------+
        v            v            v            v
     NIST AI RMF  ISO 42001   sector rule   customer DDQ

   the alternative is a control set per regulator, forever
```

Voluntary frameworks make a better spine than regulation, for two reasons that
have nothing to do with enthusiasm for standards.

**They are written as controls.** NIST AI RMF and ISO 42001 describe things you
*do*. Regulation describes outcomes you must achieve, which is harder to
operationalise and easier to satisfy on paper.

**They change more slowly than the law.** Building against a framework and
mapping outward to regulation means new regulation is a mapping exercise rather
than a programme.

The method: pick one spine with the best coverage of the controls you actually
operate, map outward, and be explicit about what the spine does **not** reach —
because every spine has gaps, and the gaps are where the sector overlay lives.

## 3 · The procedure, as a skill

The skill computes coverage per framework against your own control catalogue, selects the widest as a spine, and supplies the remaining three controls from the others — then costs that against building a programme per framework.

In [ ]:
# skills/regulatory/framework-spine-selection/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: framework-spine-selection
description: >-
  Choose the framework that covers the most of your controls as a spine, supply
  the remainder from the others, and compare that against building a programme
  per framework. Use when several frameworks apply and each is proposing its own
  programme.
allowed-tools: Read, Grep, Glob
---

# One spine, and the rest as supplements

Voluntary frameworks overlap heavily. Building a programme per framework
multiplies the same work by the number of logos, and the artefacts are nearly
identical. Choosing the one with the widest coverage as a spine and supplying
the gaps from the others produces the same coverage for a fraction of the
effort — and the comparison is arithmetic, so it survives a meeting.

## When to use this

When more than one framework is in scope, and when a second framework is
proposed as a separate workstream.

## Procedure

**1 — Take your control catalogue as the fixed thing.** Frameworks are mapped to
it, not the other way round. If you do not have one, build it first; there is
nothing to compare otherwise.

**2 — Compute coverage per framework.** How many of your controls each one
addresses. Report the counts, not impressions of comprehensiveness.

**3 — Select the widest as the spine,** and name the controls it leaves
uncovered. Those are the gaps, and they are usually few.

**4 — Assign each gap to whichever framework covers it best.** One control, one
source. The result is a single programme with a few supplements rather than
several programmes.

**5 — Cost both plans.** Spine-plus-supplements against per-framework. Include
the duplicated evidence collection, which is where the difference actually sits.

## Output contract

```json
{
  "catalogue": ["str"],
  "coverage": [{"framework": "str", "covers": ["str"], "count": 0}],
  "spine": {"framework": "str", "count": 0, "gaps": ["str"]},
  "supplements": [{"control": "str", "from": "str"}],
  "cost": {"spine_plan": 0, "per_framework_plan": 0, "duplicated_evidence": 0}
}
```

## Failure modes

- **Choosing the spine by reputation.** Choose it by coverage of your controls.
- **Building per framework.** The evidence is duplicated, not the assurance.
- **No control catalogue.** There is nothing to map against.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/regulatory/framework-spine-selection/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/regulatory/framework-spine-selection/scripts/framework_spine_selection.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Pick the framework that covers the most controls as a spine, and supply the remainder from the others rather than building per-framework.

This is the executable half of the `framework-spine-selection` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from collections import defaultdict

CONTROLS = {
 "AC-1": ("NIST AI RMF: GOVERN-1.2", "ISO 42001: 6.1", "EU AI Act: Art.14"),
 "AC-2": ("NIST AI RMF: MANAGE-2.2", "ISO 42001: 8.1"),
 "SB-1": ("NIST AI RMF: MANAGE-2.1", "ISO 27001: A.8.20"),
 "SB-2": ("EU AI Act: Art.14",),
 "EV-1": ("ISO 42001: 9.1", "EU AI Act: Art.12"),
 "EV-2": ("NIST AI RMF: MEASURE-2.3",),
 "DR-1": ("NIST AI RMF: MEASURE-2.4", "ISO 42001: 9.1"),
 "ST-1": ("EU AI Act: Art.14", "DORA: Art.11"),
}
by_fw = defaultdict(set)
for cid, fws in CONTROLS.items():
    for f in fws:
        by_fw[f.split(":")[0]].add(cid)

print(f"{'framework':18s}{'covers':>8}  controls")
print("-" * 62)
for fw, cids in sorted(by_fw.items(), key=lambda kv: -len(kv[1])):
    print(f"{fw:18s}{len(cids):>8}  {sorted(cids)}")

spine = max(by_fw, key=lambda f: len(by_fw[f]))
gaps = sorted(set(CONTROLS) - by_fw[spine])
print(f"\nbest spine: {spine} covering {len(by_fw[spine])}/{len(CONTROLS)}")
print(f"not reached by the spine: {gaps}")

def per_regulation(controls):
    """Build a separate control set for each instrument. The usual approach."""
    sets = defaultdict(set)
    for cid, fws in controls.items():
        for f in fws:
            sets[f.split(":")[0]].add(cid)
    return sets

sets = per_regulation(CONTROLS)
total_implementations = sum(len(v) for v in sets.values())
distinct_controls = len(CONTROLS)
print(f"distinct controls actually needed : {distinct_controls}")
print(f"control implementations if built per-framework : {total_implementations}")
print(f"duplication factor : {total_implementations/distinct_controls:.1f}×")

print("\ncontrols claimed by more than one framework:")
for cid, fws in CONTROLS.items():
    if len(fws) > 1:
        print(f"   {cid}  {len(fws)} frameworks: {[f.split(':')[0] for f in fws]}")
print("\nBuilt separately, these drift: the ISO version of AC-1 and the AI Act")
print("version diverge, evidence is produced twice, and neither is trusted.")
assert total_implementations > distinct_controls

def spine_plan(controls, spine):
    covered = {c for c, fws in controls.items() if any(f.startswith(spine) for f in fws)}
    gaps = sorted(set(controls) - covered)
    secondary = defaultdict(list)
    for g in gaps:
        for f in controls[g]:
            secondary[f.split(":")[0]].append(g)
    return {"spine": spine, "covered": sorted(covered), "gaps": gaps,
            "secondary_sources": {k: v for k, v in secondary.items()
                                  if not k.startswith(spine)}}

plan = spine_plan(CONTROLS, "NIST AI RMF")
print(f"spine              {plan['spine']}")
print(f"covered by spine   {len(plan['covered'])}/{len(CONTROLS)}  {plan['covered']}")
print(f"gaps               {plan['gaps']}")
print("secondary sources needed for the gaps:")
for fw, cids in plan["secondary_sources"].items():
    print(f"   {fw:20s} supplies {cids}")

print("\nstatement for the assessor:")
print(f"   'We operate {len(CONTROLS)} AI controls, built against {plan['spine']}.")
print(f"    {len(plan['covered'])} map directly to it; {len(plan['gaps'])} come from")
print(f"    {list(plan['secondary_sources'])}. Each control produces one artefact,")
print("    which satisfies every clause it maps to.'")
assert plan["gaps"]

## What you just proved

NIST AI RMF covers the most controls (4 of 8) and is selected as the spine, leaving SB-2, EV-1 and ST-1 as gaps supplied by ISO 42001, the EU AI Act and DORA. Building per-framework would produce 14 control implementations for 8 distinct controls — a 1.8× duplication factor with controls claimed by several frameworks drifting apart.

## Your turn

Pick your spine and justify it in one sentence to an assessor. "It has the best coverage of the controls we actually operate" is far stronger than "it is the one our regulator mentioned".

---

**Next → [E2.4 · Sector overlays](https://spbreed.github.io/cyber-commons/lessons/E2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*